In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("Incremental Flag","0")
increm_flag=dbutils.widgets.get("Incremental Flag")
print(increm_flag)

In [0]:
df_labresult_src=spark.sql('''
                     select * from  parquet.`abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/lab_result`''')

In [0]:
df_labresult_src.limit(5).display()

In [0]:
df_dim_patient=spark.sql('SELECT * FROM healthcare.gold.dim_patient')
df_dim_provider=spark.sql('SELECT * FROM healthcare.gold.dim_provider')
df_dim_labtest=spark.sql('SELECT * FROM healthcare.gold.dim_labtest')
df_fact_encounter=spark.sql('SELECT * FROM healthcare.gold.fact_encounter')
df_dim_date=spark.sql('SELECT * FROM healthcare.gold.dim_date')

In [0]:
df_labresult_src.count()

In [0]:
df_staged_fact=df_labresult_src.join(df_dim_patient,df_labresult_src["mrn"]==df_dim_patient["mrn"],"inner")\
    .join(df_dim_provider,df_labresult_src["ordering_npi"]==df_dim_provider["npi"],"inner")\
        .join(df_dim_labtest,df_labresult_src["loinc_code"]==df_dim_labtest["loinc_code"],"inner")\
                .join(df_fact_encounter,df_labresult_src["encounter_id"]==df_fact_encounter["encounter_id"],"inner")\
                    .join(df_dim_date,df_labresult_src["result_date"]==df_dim_date["full_date"],"inner")\
                        .select(df_dim_patient["dim_patient_key"],df_dim_provider["dim_provider_key"],df_dim_labtest["dim_labtest_key"],df_fact_encounter["encounter_key"],df_dim_date["dim_date_key"],df_labresult_src["order_id"],df_labresult_src["result_value_numeric"],df_labresult_src["result_value_text"],df_labresult_src["abnormal_flag"],df_labresult_src["result_unit"])

In [0]:
df_staged_fact.count()

In [0]:
df_staged_fact.limit(5).display()

In [0]:
if increm_flag=='0':
    max_value=1
else:
    max_value_df=spark.sql('''
            select coalesce(max(labresult_key),0) from healthcare.gold.fact_labresult''')
    max_value=max_value_df.collect()[0][0]+1

In [0]:
if spark.catalog.tableExists('healthcare.gold.fact_labresult'):
    df_sink=spark.sql('''
        select labresult_key,dim_patient_key, 
dim_provider_key, 
dim_labtest_key, 
encounter_key, 
dim_date_key, 
order_id,
result_value_numeric,
result_value_text,
abnormal_flag,
result_unit
 from healthcare.gold.fact_labresult''')
else:
    df_sink=spark.sql('''
            select 1 as labresult_key,
order_id,
result_value_numeric,
result_value_text,
abnormal_flag,
result_unit from parquet.`abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/lab_result` where 1=0''')

In [0]:
df_sink.limit(5).display()

In [0]:
df_sink.count()

In [0]:
df_filter=df_staged_fact.join(df_sink,df_staged_fact["order_id"]==df_sink["order_id"],"left")\
    .select(df_staged_fact["*"],df_sink["labresult_key"])

In [0]:
df_filter.limit(5).display()


In [0]:
df_new_rec=df_filter.filter(df_filter["labresult_key"].isNull())
df_existing_rec=df_filter.filter(df_filter["labresult_key"].isNotNull())
df_new_rec.limit(5).display()
df_existing_rec.limit(5).display()

In [0]:
dupes = df_existing_rec.groupBy("labresult_key").count().filter("count > 1")
dupes.show()

#the duplicates
df_existing_rec.join(dupes.select("labresult_key"), "labresult_key") \
    .orderBy("labresult_key") \
    .display()

In [0]:
windowSpec = Window.orderBy("order_id")
df_insert = df_new_rec.withColumn(
    "labresult_key", row_number().over(windowSpec) + lit(max_value - 1)
)

In [0]:
df_insert.limit(5).display()

In [0]:
from delta.tables import DeltaTable
if spark.catalog.tableExists("healthcare.gold.fact_labresult"):
    #insert the new entries
    df_insert.write.format("delta").mode("append").saveAsTable("healthcare.gold.fact_labresult")
    #update the entries
    deltaTable=DeltaTable.forName(spark,"healthcare.gold.fact_labresult")
    df_existing_rec_dedup = df_existing_rec.dropDuplicates(["labresult_key"])
    deltaTable.alias("target").merge(df_existing_rec_dedup.alias("source"),"target.labresult_key = source.labresult_key")\
        .whenMatchedUpdate(set={
            "dim_patient_key": "source.dim_patient_key",
            "dim_provider_key": "source.dim_provider_key",
            "dim_labtest_key": "source.dim_labtest_key",
            "dim_date_key": "source.dim_date_key",
            "encounter_key": "source.encounter_key",
            "order_id": "source.order_id",
            "result_value_numeric": "source.result_value_numeric",
            "result_value_text": "source.result_value_text",
            "abnormal_flag": "source.abnormal_flag",
            "result_unit": "source.result_unit"
    }).execute()
else:
    df_insert.write.format("delta").mode("append").saveAsTable("healthcare.gold.fact_labresult")

In [0]:
%sql
select * from healthcare.gold.fact_labresult